# TQE — HAGCS Six-Stage Pipeline on IBM Quantum Hardware

One notebook for all six stages of the TQE study, run on the **real IBM Quantum processors `ibm_kingston` and `ibm_marrakesh`** (Heron r2). No fake backends or noisy simulators stand in for hardware; the only simulation is the noise-free ideal reference of Stage 2.

### Experimental scope
- configuration c = (b, o, r, m): processor b, oracle realization o, Grover iterations r (= k), error-management strategy m
- problem sizes **n = 5, 6** (N = 32, 64), M = 3 satisfying assignments each
- oracles: **Oracle A (direct marked-state)** and **Oracle B (clause-structured 2-SAT)**; iterations **k = 5, 6**
- strategies: **None (RAW), DD, PT, ZNE, TREX**, each with readout correction; ALL4 as sensitivity only
- HAGCS: Pareto-based feasibility filtering, then uncertainty-aware, cost-sensitive ranking
- Stage 6: **fresh** QPU jobs of the HAGCS-selected configuration and method

### How to run
1. Run all cells. With an IBM Quantum account (`IBM_QUANTUM_TOKEN` / saved account), Stage 3 downloads the raw counts of the completed jobs `db1l4n9b694s73dsemrg` (Kingston) and `db1l4p6egvvc73bhpq40` (Marrakesh) and caches them.
2. Set `SUBMIT_STAGE4_JOBS = True` and run once to submit the Stage-4 campaign (one job per processor × strategy). Save the printed job IDs; put them in `JOB_ID_OVERRIDES` if you continue in another session or on another machine. Set the flag back to `False`.
3. When the Stage-4 jobs are DONE, rerun all cells. Stage 5 selects one configuration per (processor, n).
4. Set `SUBMIT_STAGE6_JOBS = True` and run once to submit the Stage-6 validation jobs, then set it back to `False` and rerun when they finish.

The final audit cell states which stage claims are supported by data. A claim is supported only when the audit says so.

### Corrections built in
1. **One bit-order convention** (below). Earlier notebook versions declared the n=6 solution as `010101` in variable order but looked it up in Qiskit count strings, so they amplified one state and counted another. Every solution string here is a Qiskit count key.
2. **Circuit-level oracle validation** on every basis state for both oracles, and an exact statevector check of every Grover circuit against the analytical formula. Earlier versions hard-coded the Oracle-B check to `True` and used a phase test with the wrong sign.
3. **Stage 3 is re-scored from raw IBM counts**, and its resources come from the circuits that actually ran.
4. **Stage 6 is independent.** The step-15 "final validation" jobs are the same jobs as the Stage-3 characterization, so they cannot validate a selection made from them. Stage 6 uses fresh jobs.
5. **Readout correction for every strategy**: calibration-matrix correction for Sampler methods (RAW/DD/PT) and TREX for Estimator methods (ZNE/TREX/ALL4).
6. **ZNE/TREX observable fixed.** The projector is built on the logical circuit and mapped with the transpiler layout. Earlier versions built it on the full device width, so `apply_layout` failed silently and the wrong physical qubits were measured.
7. **QPU cost and time-to-solution** come from measured quantum seconds and from two-qubit-gate executions.
8. **HAGCS order follows the pipeline:** Pareto feasibility first, then robust ranking. The HAGCS-vs-max-P_valid trade-off is reported separately from Stage 6.

### Bit-order convention (used everywhere)
Variable x_i of the 2-SAT instance (1-based, as in the clause lists) is stored on **data qubit i−1**. Every solution string is written in **Qiskit count-string order q_{n-1}…q_0**, so the rightmost character is x_1. These strings are exactly the keys of Sampler counts. The variable-order assignment (x1…xn) is the reversed string and is reported alongside for readability.

### Six pipeline stages
1. Circuit and configuration space
2. Ideal simulation
3. 2-SAT instances on hardware (raw characterization)
4. Error suppression and mitigation
5. HAGCS selection
6. Independent QPU validation of the selected configuration

In [ ]:
# ============================================================
# 1. MASTER CONFIGURATION — SCIENTIFIC CONTROL
# ============================================================
from pathlib import Path
import os, json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2024
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

BACKENDS = ["ibm_kingston", "ibm_marrakesh"]
N_VALUES = [5, 6]
ORACLES = ["A", "B"]
K_VALUES = [5, 6]
M_SOLUTIONS = 3

# The completed reference campaign used 500 shots. Keep the nominal
# budget matched for new raw/suppression runs and state mitigation
# overhead explicitly rather than changing the nominal budget.
SHOTS = 500
INDEPENDENT_SHOTS = 500

METHODS = ["RAW", "DD", "PT", "ZNE", "TREX", "ALL4"]
PRIMARY_METHODS = ["RAW", "DD", "PT", "ZNE", "TREX"]   # ALL4 = sensitivity only
SUPPRESSION_METHODS = {"RAW", "DD", "PT"}
ESTIMATOR_METHODS = {"ZNE", "TREX", "ALL4"}
METHOD_LABELS = {"RAW": "None", "DD": "DD", "PT": "PT", "ZNE": "ZNE", "TREX": "TREX", "ALL4": "ALL4"}

ZNE_NOISE_FACTORS = [1, 3, 5]
ZNE_EXTRAPOLATOR = "exponential"
TREX_RANDOMIZATIONS = 32
TWIRL_RANDOMIZATIONS = "auto"

# Readout correction applied to every method (pipeline: "each with readout correction").
#   Sampler methods (RAW/DD/PT): tensored calibration-matrix correction from
#   |0...0> and |1...1> calibration circuits on the same physical qubits, in the same job.
#   Estimator methods (ZNE/TREX/ALL4): TREX measurement mitigation.
READOUT_CORRECTION = True
CAL_SHOTS = 2000
BOOTSTRAP_SAMPLES = 400

# HAGCS selection controls.
# Pareto objectives per (backend, n): maximize the uncertainty-aware lower bound
# P_valid - 2*sigma, minimize 2Q gates, depth and QPU cost.
HAGCS_PARETO_OBJECTIVES = {"P_lower_2sigma": "max", "two_qubit_gates": "min",
                           "depth": "min", "qpu_cost_2q_executions": "min"}
# If True, a configuration is feasible only if its 95% CI lies above the
# random-guessing baseline M/N. Reported as a flag either way.
REQUIRE_ABOVE_RANDOM = False

# Real-QPU submission switches. Each is OFF by default; turn one on, run once, turn it off.
SUBMIT_STAGE4_JOBS = False      # Stage-4 strategy campaign (one job per processor x strategy)
SUBMIT_STAGE6_JOBS = False      # Stage-6 fresh validation of the HAGCS selection
# Strategies submitted in Stage 4. ALL4 is sensitivity analysis; drop it to save QPU time.
STAGE4_METHODS = ["RAW", "DD", "PT", "ZNE", "TREX", "ALL4"]
# Job IDs from another session/machine, e.g. {"ibm_kingston/ZNE": "d1abc...",
# "stage6_ibm_kingston/RAW": "d1xyz..."}. Used instead of the job-ID files.
JOB_ID_OVERRIDES = {}

# Until a readout-corrected RAW run completes, the Stage-3 raw characterization
# fills the RAW cells of the strategy matrix (flagged readout_corrected=False).
USE_STAGE3_AS_RAW_BASELINE = True

ROOT = Path("results/TQE_HAGCS_six_stage")
FIG_DIR = ROOT / "figures"
TAB_DIR = ROOT / "tables"
JOB_DIR = ROOT / "jobs"
CIRCUIT_DIR = ROOT / "circuits"
for p in [ROOT, FIG_DIR, TAB_DIR, JOB_DIR, CIRCUIT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

print("TQE six-stage configuration loaded.")
print("Nominal shots:", SHOTS, "| readout correction:", READOUT_CORRECTION)

## Stage 1 — Circuit and configuration space

The algorithmic space is

\[\mathcal C = \{5,6\}\times\{A,B\}\times\{5,6\},\]

which gives 8 configurations per processor. The satisfying assignments are enumerated from the clauses, and both oracle **circuits** are then checked on every computational basis state before any simulation or hardware analysis.

In [ ]:
# ============================================================
# 2. 2-SAT INSTANCES + CLAUSE-LEVEL ENUMERATION
# ============================================================
# Convention: variable x_i (1-based) lives on data qubit i-1; every solution
# string is a Qiskit count key (q_{n-1}...q_0), so the RIGHTMOST character is x_1.
KSAT = {
    5: {
        "clauses": [[-1, 3], [-3, 5], [-5, 1], [-2, 4], [-4, 2], [-1, 2]],
        "solutions": ["00000", "01010", "11111"],
    },
    6: {
        "clauses": [[-1, 3], [-3, 5], [-5, 1], [-2, 4], [-4, 6], [-6, 2], [-1, 2]],
        # Assignment x = (0,1,0,1,0,1) -> Qiskit count key "101010".
        "solutions": ["000000", "101010", "111111"],
    },
}

def var_value(bits, v):
    """Value of variable x_v (1-based) in a Qiskit-ordered bitstring."""
    return int(bits[len(bits) - v])

def assignment_string(bits):
    """Qiskit count key -> variable-order string x1...xn (for reporting only)."""
    return bits[::-1]

def clause_satisfied(bits, clause):
    return any(var_value(bits, abs(lit)) == (1 if lit > 0 else 0) for lit in clause)

def satisfies_2sat(bits, clauses):
    return all(clause_satisfied(bits, c) for c in clauses)

def exhaustive_solutions(n, clauses):
    return [
        format(i, f"0{n}b")
        for i in range(2**n)
        if satisfies_2sat(format(i, f"0{n}b"), clauses)
    ]

validation_rows = []
for n in N_VALUES:
    expected = exhaustive_solutions(n, KSAT[n]["clauses"])
    declared = sorted(KSAT[n]["solutions"])
    assert expected == declared, (n, expected, declared)
    assert len(expected) == M_SOLUTIONS, (n, expected)
    validation_rows.append({
        "n": n,
        "N": 2**n,
        "solutions_qiskit_order": ";".join(expected),
        "solutions_variable_order_x1_to_xn": ";".join(assignment_string(s) for s in expected),
        "M": len(expected),
        # Set by the circuit-level check in the next-but-one cell; never hard-coded.
        "oracle_A_circuit_equals_2SAT": None,
        "oracle_B_circuit_equals_2SAT": None,
    })

truth_table_validation = pd.DataFrame(validation_rows)
print("Clause enumeration matches declared solutions.")
display(truth_table_validation)

CONFIGS = [
    {"n": n, "oracle": o, "k": k}
    for n in N_VALUES for o in ORACLES for k in K_VALUES
]
config_df = pd.DataFrame(CONFIGS)
config_df["N"] = 2 ** config_df["n"]
config_df["M"] = M_SOLUTIONS
config_df["config_id"] = (
    "n" + config_df["n"].astype(str)
    + "_O" + config_df["oracle"]
    + "_k" + config_df["k"].astype(str)
)
display(config_df)
config_df.to_csv(TAB_DIR / "stage1_configuration_space.csv", index=False)

In [ ]:
# ============================================================
# 3. CIRCUIT BUILDERS — ORACLE A AND ORACLE B
# ============================================================
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import MCXGate

def mcz(qc, qubits):
    *controls, target = qubits
    if not controls:
        qc.z(target)
        return
    qc.h(target)
    qc.append(MCXGate(len(controls)), controls + [target])
    qc.h(target)

def diffuser(n):
    qc = QuantumCircuit(n, name="Diffuser")
    qc.h(range(n))
    qc.x(range(n))
    mcz(qc, list(range(n)))
    qc.x(range(n))
    qc.h(range(n))
    return qc

def oracle_marked(n, states):
    """Oracle A: direct phase marking of the satisfying assignments.

    `states` are Qiskit count keys; character position j from the right is qubit j.
    """
    qc = QuantumCircuit(n, name="Oracle_A")
    for s in states:
        zero_positions = [i for i, bit in enumerate(reversed(s)) if bit == "0"]
        if zero_positions:
            qc.x(zero_positions)
        mcz(qc, list(range(n)))
        if zero_positions:
            qc.x(zero_positions)
    return qc

def oracle_clauses(n, clauses):
    """Oracle B: clause-structured 2-SAT phase oracle.

    Variable x_v is data qubit v-1 (same convention as KSAT solution strings).
    Each ancilla records clause satisfaction; a multi-controlled X on the
    output qubit fires only when all clauses are satisfied. The output is
    prepared in |-> by ksat_circuit, so the X becomes a phase kickback.
    Ancillas are uncomputed.
    """
    x = QuantumRegister(n, "x")
    a = QuantumRegister(len(clauses), "c")
    o = QuantumRegister(1, "out")
    qc = QuantumCircuit(x, a, o, name="Oracle_B")

    def mark_clause_satisfied(j, clause):
        # A clause is false only when every positive literal is 0 and every
        # negative literal is 1. X on positive variables converts that
        # falsifying pattern to all-ones controls.
        positive = [abs(lit) - 1 for lit in clause if lit > 0]
        if positive:
            qc.x([x[i] for i in positive])
        qc.append(
            MCXGate(len(clause)),
            [x[abs(lit) - 1] for lit in clause] + [a[j]],
        )
        if positive:
            qc.x([x[i] for i in positive])
        # The MCX above marks the falsifying pattern; invert it so ancilla=1
        # means clause satisfied.
        qc.x(a[j])

    for j, clause in enumerate(clauses):
        mark_clause_satisfied(j, clause)

    qc.append(MCXGate(len(clauses)), list(a) + [o[0]])

    # Exact uncomputation in reverse order.
    for j, clause in reversed(list(enumerate(clauses))):
        # Undo satisfied marker.
        qc.x(a[j])
        positive = [abs(lit) - 1 for lit in clause if lit > 0]
        if positive:
            qc.x([x[i] for i in positive])
        qc.append(
            MCXGate(len(clause)),
            [x[abs(lit) - 1] for lit in clause] + [a[j]],
        )
        if positive:
            qc.x([x[i] for i in positive])

    return qc

def ksat_circuit(n, oracle, k, measure=True):
    data = KSAT[n]

    if oracle == "A":
        qc = QuantumCircuit(n)
        qc.h(range(n))
        ob = oracle_marked(n, data["solutions"])
        dif = diffuser(n)
        for _ in range(k):
            qc.compose(ob, inplace=True)
            qc.compose(dif, inplace=True)
        if measure:
            cr = ClassicalRegister(n, "meas")
            qc.add_register(cr)
            qc.measure(range(n), cr)
        return qc

    x = QuantumRegister(n, "x")
    a = QuantumRegister(len(data["clauses"]), "c")
    o = QuantumRegister(1, "out")
    qc = QuantumCircuit(x, a, o)
    qc.x(o)
    qc.h(o)
    qc.h(x)

    ob = oracle_clauses(n, data["clauses"])
    dif = diffuser(n)

    for _ in range(k):
        qc.compose(ob, inplace=True)
        qc.compose(dif, qubits=list(x), inplace=True)

    qc.h(o)
    qc.x(o)

    if measure:
        cr = ClassicalRegister(n, "meas")
        qc.add_register(cr)
        qc.measure(x, cr)
    return qc

def solution_states(n):
    return KSAT[n]["solutions"]

print("Circuit builders ready.")

In [ ]:
# ============================================================
# 4. MANDATORY CIRCUIT-LEVEL ORACLE VALIDATION
# ============================================================
# Each oracle circuit is applied to every computational basis state of the data
# register. A correct phase oracle returns the same basis state (all ancillas and
# the output qubit restored to |0>) multiplied by -1 if the state satisfies the
# 2-SAT instance and +1 otherwise. For Oracle B the output qubit is prepared in
# |-> and un-prepared afterwards, exactly as in ksat_circuit.
from qiskit.quantum_info import Statevector

def oracle_phase(n, oracle, state):
    if oracle == "A":
        nq = n
        body = oracle_marked(n, solution_states(n))
    else:
        clauses = KSAT[n]["clauses"]
        nq = n + len(clauses) + 1
        body = oracle_clauses(n, clauses)
    qc = QuantumCircuit(nq)
    for q, bit in enumerate(reversed(state)):
        if bit == "1":
            qc.x(q)
    if oracle == "B":
        qc.x(nq - 1); qc.h(nq - 1)
    qc.compose(body, qubits=list(range(nq)), inplace=True)
    if oracle == "B":
        qc.h(nq - 1); qc.x(nq - 1)
    sv = Statevector.from_instruction(qc).data
    idx = int(state, 2)            # ancillas and output back in |0>
    amp = sv[idx]
    # All amplitude must remain on the input basis state.
    assert np.isclose(abs(amp), 1.0, atol=1e-8), (n, oracle, state, amp)
    return float(np.real(amp))

for n in N_VALUES:
    sols = set(solution_states(n))
    for oracle in ORACLES:
        mismatches = []
        for state in [format(i, f"0{n}b") for i in range(2**n)]:
            expected = -1.0 if state in sols else 1.0
            if not np.isclose(oracle_phase(n, oracle, state), expected, atol=1e-8):
                mismatches.append(state)
        assert not mismatches, f"Oracle {oracle} n={n} marks the wrong states: {mismatches}"
        truth_table_validation.loc[truth_table_validation.n == n, f"oracle_{oracle}_circuit_equals_2SAT"] = True

display(truth_table_validation)
truth_table_validation.to_csv(TAB_DIR / "stage1_oracle_truth_table_validation.csv", index=False)
print("Circuit-level Oracle A/B phase validation PASSED for every basis state.")

In [ ]:
# ============================================================
# 5. IDEAL GROVER MODEL + RESOURCE ACCOUNTING
# ============================================================
def ideal_grover_probability(n, k, M=M_SOLUTIONS):
    N = 2 ** n
    theta = np.arcsin(np.sqrt(M / N))
    return float(np.sin((2 * k + 1) * theta) ** 2)

def random_baseline(n, M=M_SOLUTIONS):
    """Success probability of uniform random guessing, M/N."""
    return M / 2 ** n

def decomposed_circuit(qc, reps=20):
    # Repeatedly decompose named/composite instructions so resource counts
    # reflect primitive one-/two-qubit operations rather than MCX/composite
    # wrappers.
    out = qc
    for _ in range(reps):
        before = len(out.data)
        out = out.decompose()
        if len(out.data) == before and all(getattr(inst.operation, "definition", None) is None
                                           for inst in out.data):
            break
    return out

NON_GATE_OPS = {"barrier", "measure", "delay", "reset"}

def count_two_qubit_ops(qc):
    count = 0
    for item in qc.data:
        if item.operation.name in NON_GATE_OPS:
            continue
        if len(item.qubits) == 2:
            count += 1
    return int(count)

def logical_resources(qc):
    dq = decomposed_circuit(qc)
    return {
        "logical_qubits": dq.num_qubits,
        "logical_depth": dq.depth(),
        "logical_2q_gates": count_two_qubit_ops(dq),
        "logical_ops": len(dq.data),
    }

stage1_rows = []
for cfg in CONFIGS:
    qc = ksat_circuit(cfg["n"], cfg["oracle"], cfg["k"], measure=False)
    r = logical_resources(qc)
    stage1_rows.append({
        **cfg,
        "config_id": f"n{cfg['n']}_O{cfg['oracle']}_k{cfg['k']}",
        "N": 2 ** cfg["n"],
        "M": M_SOLUTIONS,
        "P_ideal": ideal_grover_probability(cfg["n"], cfg["k"]),
        "P_random": random_baseline(cfg["n"]),
        **r,
    })

stage1_df = pd.DataFrame(stage1_rows)
display(stage1_df)
stage1_df.to_csv(TAB_DIR / "stage1_circuit_resources.csv", index=False)

In [ ]:
# ============================================================
# 6. PAPER-READY LOGICAL CIRCUIT DIAGRAMS — NO QPU TIME
# ============================================================
DIAG_DIR = ROOT / "circuit_diagrams"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

REPRESENTATIVE = [(5, "A", 5), (5, "B", 5), (6, "A", 6), (6, "B", 6)]

def save_circuit_figure(qc, stem, title=None, fold=-1, idle_wires=True):
    try:
        fig = qc.draw(output="mpl", fold=fold, scale=0.55, idle_wires=idle_wires)
        if title:
            fig.suptitle(title, fontsize=13, y=1.02, fontweight="bold")
        fig.savefig(DIAG_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
        fig.savefig(DIAG_DIR / f"{stem}.pdf", bbox_inches="tight")
        plt.close(fig)
        return True
    except Exception as exc:
        print(f"Diagram generation failed for {stem}: {exc}")
        return False

for n, oracle, k in REPRESENTATIVE:
    save_circuit_figure(ksat_circuit(n, oracle, k), f"logical_n{n}_oracle{oracle}_k{k}",
                        title=f"K-SAT Grover circuit: n={n}, Oracle {oracle}, k={k}")
# One Grover iteration of each oracle, for the methods figure.
for n in N_VALUES:
    save_circuit_figure(ksat_circuit(n, "A", 1), f"logical_n{n}_oracleA_one_iteration",
                        title=f"One Grover iteration, Oracle A, n={n}")
    save_circuit_figure(ksat_circuit(n, "B", 1), f"logical_n{n}_oracleB_one_iteration",
                        title=f"One Grover iteration, Oracle B, n={n}")
print("Logical circuit diagrams saved to", DIAG_DIR)

## Stage 2 — Ideal simulation

The ideal reference is calculated analytically and checked twice: (i) **exactly** with `Statevector` (no shot noise; must agree with the formula to 1e-6, otherwise the notebook stops), and (ii) with a finite-shot Aer run when available. The analytical value remains the reference for hardware-loss and relative-to-ideal calculations.

Note that for n=5, k=5 the ideal probability (0.077) is *below* random guessing (3/32 = 0.094): k=5 overshoots the optimum r=2. Hardware values at or above this ideal value at k=5 therefore indicate depolarisation toward the uniform distribution, not amplification.

In [ ]:
# ============================================================
# 7. IDEAL SIMULATOR / ANALYTICAL REFERENCE
# ============================================================
AER_AVAILABLE = False
try:
    from qiskit_aer import AerSimulator
    AER_AVAILABLE = True
except Exception as exc:
    print("qiskit-aer is not available; exact statevector + analytical reference will be used.")
    print("Reason:", exc)

ideal_rows = []
simulator = AerSimulator(method="statevector") if AER_AVAILABLE else None

for cfg in CONFIGS:
    n, oracle, k = cfg["n"], cfg["oracle"], cfg["k"]
    sols = solution_states(n)
    p_analytic = ideal_grover_probability(n, k)

    # Exact check: marginal probabilities of the data register.
    probs = Statevector(ksat_circuit(n, oracle, k, measure=False)).probabilities_dict(qargs=list(range(n)))
    p_exact = float(sum(probs.get(s, 0.0) for s in sols))
    assert abs(p_exact - p_analytic) < 1e-6, (
        f"Circuit n={n} O{oracle} k={k}: exact P={p_exact:.6f} != analytic {p_analytic:.6f}")

    row = {**cfg, "backend": "ideal_simulator", "P_ideal": p_analytic,
           "P_exact_statevector": p_exact, "P_random": random_baseline(n),
           **{f"Pexact_{s}": probs.get(s, 0.0) for s in sols}}

    if AER_AVAILABLE:
        qc = ksat_circuit(n, oracle, k, measure=True)
        counts = simulator.run(qc, shots=SHOTS, seed_simulator=SEED).result().get_counts()
        total = sum(counts.values())
        pvalid = sum(counts.get(s, 0) for s in sols) / total
        sigma = math.sqrt(p_analytic * (1 - p_analytic) / total)
        row.update({"method": "IDEAL_AER", "shots": total, "P_valid": pvalid,
                    "analytic_minus_AER": p_analytic - pvalid,
                    "AER_z_score": (pvalid - p_analytic) / sigma if sigma > 0 else np.nan})
    else:
        row.update({"method": "IDEAL_EXACT", "shots": np.nan, "P_valid": p_exact,
                    "analytic_minus_AER": np.nan, "AER_z_score": np.nan})
    row["expected_shots"] = 1.0 / p_analytic if p_analytic > 0 else np.inf
    ideal_rows.append(row)

ideal_df = pd.DataFrame(ideal_rows)
ideal_df["hardware_loss"] = 0.0
ideal_df["relative_to_ideal"] = 1.0
display(ideal_df)
ideal_df.to_csv(TAB_DIR / "stage2_ideal_simulation.csv", index=False)
print("Exact statevector matches the analytical Grover formula for all 8 configurations.")
if AER_AVAILABLE and (ideal_df["AER_z_score"].abs() > 4).any():
    warnings.warn("An Aer finite-shot estimate deviates by more than 4 sigma from the analytical value.")

In [ ]:
# ============================================================
# 8. CONNECT TO IBM QUANTUM (REAL PROCESSORS ONLY)
# ============================================================
# Credentials are read from the environment or from an account saved with
# QiskitRuntimeService.save_account(...). Never paste a token into the notebook.
#   IBM_QUANTUM_TOKEN     (optional) API key; saved as the default account if set
#   IBM_QUANTUM_INSTANCE  (optional) instance CRN
from qiskit.transpiler import generate_preset_pass_manager

IBM_AVAILABLE = False
service = None
backend_objects = {}
try:
    from qiskit_ibm_runtime import QiskitRuntimeService
    token = os.getenv("IBM_QUANTUM_TOKEN")
    instance = os.getenv("IBM_QUANTUM_INSTANCE")
    if token:
        QiskitRuntimeService.save_account(token=token, instance=instance, channel="ibm_quantum_platform",
                                          set_as_default=True, overwrite=True)
        print("IBM Quantum account saved from IBM_QUANTUM_TOKEN.")
    service = (QiskitRuntimeService(channel="ibm_quantum_platform", instance=instance)
               if instance else QiskitRuntimeService())
    for name in BACKENDS:
        b = service.backend(name)
        # Guard: only real IBM Quantum processors are accepted.
        assert type(b).__name__ == "IBMBackend" and not b.name.startswith("fake"), f"{name} is not a real IBM backend"
        backend_objects[name] = b
        st = b.status()
        print(f"{name}: {b.num_qubits} qubits, operational={st.operational}, pending_jobs={st.pending_jobs}, status={st.status_msg}")
    IBM_AVAILABLE = len(backend_objects) == len(BACKENDS)
except Exception as exc:
    print("IBM Quantum is not reachable from this kernel:", exc)

if not IBM_AVAILABLE:
    print("Analysis-only mode: cached results of completed IBM jobs are used; nothing is simulated "
          "in place of hardware, and no new jobs can be submitted.")
if (SUBMIT_STAGE4_JOBS or SUBMIT_STAGE6_JOBS) and not IBM_AVAILABLE:
    raise RuntimeError("Submission requested but the IBM Quantum processors are not available.")

_PASS_MANAGERS = {}
def pass_manager(bname):
    if bname not in _PASS_MANAGERS:
        _PASS_MANAGERS[bname] = generate_preset_pass_manager(
            backend=backend_objects[bname], optimization_level=3, seed_transpiler=SEED)
    return _PASS_MANAGERS[bname]

_ISA_CACHE = {}
def isa_circuit(bname, n, oracle, k, measure):
    """Transpile once per (backend, config, measure) so resources and jobs use the same circuit."""
    key = (bname, n, oracle, k, measure)
    if key not in _ISA_CACHE:
        _ISA_CACHE[key] = pass_manager(bname).run(ksat_circuit(n, oracle, k, measure=measure))
    return _ISA_CACHE[key]

def data_physical_qubits(isa_qc, n):
    """Physical qubits holding data qubits 0..n-1 at the end of the circuit."""
    return [int(p) for p in isa_qc.layout.final_index_layout()[:n]]

def active_qubits(qc):
    return len({q for inst in qc.data if inst.operation.name != "barrier" for q in inst.qubits})

def job_quantum_seconds(job):
    getters = (lambda j: j.usage(),
               lambda j: j.metrics()["usage"]["quantum_seconds"],
               lambda j: j.metrics()["usage"]["seconds"])
    for g in getters:
        try:
            v = g(job)
            if v is not None:
                return float(v)
        except Exception:
            pass
    return np.nan

## Stage 3 — 2-SAT instances on hardware (raw characterization)

Stage 3 is the completed raw campaign on the real processors: one bundled 8-circuit Sampler job per processor (n = 5, 6 × Oracle A, B × k = 5, 6; 500 shots; no suppression or mitigation). These jobs are **characterization data**. They feed Stage 4 (as the uncorrected baseline) and Stage 5 (selection), and therefore **cannot** serve as the Stage-6 validation.

When IBM Quantum is reachable, the raw counts are downloaded from the jobs, cached to `jobs/stage3_counts.json`, and **re-scored here** against the solution set defined in Stage 1. The two-qubit count and depth are taken from the circuits that actually ran (`job.inputs`). Without a connection, the cached counts are used; if no cache exists yet, the values recorded in the original retrieval notebook are used and labelled as such.

In [ ]:
# ============================================================
# 9. STAGE 3 — RAW HARDWARE CHARACTERIZATION FROM IBM JOBS
# ============================================================
STAGE3_JOB_IDS = {
    "ibm_kingston": "db1l4n9b694s73dsemrg",
    "ibm_marrakesh": "db1l4p6egvvc73bhpq40",
}
# PUB order used when the Stage-3 jobs were submitted.
STAGE3_CONFIG_ORDER = [(5, "A", 5), (5, "A", 6), (5, "B", 5), (5, "B", 6),
                       (6, "A", 5), (6, "A", 6), (6, "B", 5), (6, "B", 6)]
STAGE3_SHOTS = 500
STAGE3_CACHE = JOB_DIR / "stage3_counts.json"

# Values printed by the original retrieval of the jobs above (500 shots each), used ONLY
# if IBM is unreachable and no cached counts exist. Per-solution probabilities follow the
# order of KSAT[n]["solutions"] (Qiskit count keys; n=6 includes "101010").
# Columns: backend, n, oracle, k, P_valid, P_sol1, P_sol2, P_sol3, 2Q gates, depth
STAGE3_RECORDED = [
    ("ibm_kingston", 5, "A", 5, 0.122, 0.040, 0.048, 0.034, 1297, 3541),
    ("ibm_kingston", 5, "A", 6, 0.126, 0.050, 0.040, 0.036, 1557, 4248),
    ("ibm_kingston", 5, "B", 5, 0.116, 0.052, 0.028, 0.036, 1791, 3643),
    ("ibm_kingston", 5, "B", 6, 0.066, 0.014, 0.028, 0.024, 2072, 4189),
    ("ibm_kingston", 6, "A", 5, 0.044, 0.016, 0.012, 0.016, 2687, 8769),
    ("ibm_kingston", 6, "A", 6, 0.028, 0.010, 0.012, 0.006, 3221, 10518),
    ("ibm_kingston", 6, "B", 5, 0.052, 0.024, 0.018, 0.010, 2365, 4337),
    ("ibm_kingston", 6, "B", 6, 0.032, 0.020, 0.006, 0.006, 3026, 5624),
    ("ibm_marrakesh", 5, "A", 5, 0.094, 0.040, 0.022, 0.032, 1297, 3516),
    ("ibm_marrakesh", 5, "A", 6, 0.076, 0.022, 0.030, 0.024, 1557, 4218),
    ("ibm_marrakesh", 5, "B", 5, 0.124, 0.044, 0.052, 0.028, 1791, 3616),
    ("ibm_marrakesh", 5, "B", 6, 0.100, 0.044, 0.028, 0.028, 2072, 4171),
    ("ibm_marrakesh", 6, "A", 5, 0.046, 0.024, 0.010, 0.012, 2687, 8771),
    ("ibm_marrakesh", 6, "A", 6, 0.052, 0.030, 0.008, 0.014, 3221, 10520),
    ("ibm_marrakesh", 6, "B", 5, 0.040, 0.020, 0.016, 0.004, 2365, 4344),
    ("ibm_marrakesh", 6, "B", 6, 0.038, 0.008, 0.018, 0.012, 3026, 5642),
]
for row in STAGE3_RECORDED:
    assert len(row) == 10, f"Malformed recorded row: {row}"
    assert abs(sum(row[5:8]) - row[4]) < 1.5e-3, f"Per-solution values do not sum to P_valid: {row}"

def _counts(pub_data):
    if hasattr(pub_data, "meas") and hasattr(pub_data.meas, "get_counts"):
        return pub_data.meas.get_counts()
    for name in dir(pub_data):
        if name.startswith("_"):
            continue
        obj = getattr(pub_data, name, None)
        if hasattr(obj, "get_counts"):
            return obj.get_counts()
    raise RuntimeError("No measurement register found in Sampler result.")

def _executed_circuits(job):
    """Circuits that actually ran, if the runtime exposes the job inputs."""
    try:
        pubs = job.inputs["pubs"]
        return [p[0] if isinstance(p, (tuple, list)) else getattr(p, "circuit", p) for p in pubs]
    except Exception:
        return None

def fetch_stage3_from_ibm():
    cache = {}
    for bname, jid in STAGE3_JOB_IDS.items():
        job = service.job(jid)
        status = str(job.status()).upper()
        print(f"Stage 3 {bname}: {jid} -> {status}")
        if "DONE" not in status and "COMPLETED" not in status:
            raise RuntimeError(f"Stage-3 job {jid} is not complete ({status}).")
        result = job.result()
        assert len(result) == len(STAGE3_CONFIG_ORDER), f"{jid}: expected {len(STAGE3_CONFIG_ORDER)} PUBs, got {len(result)}"
        circuits = _executed_circuits(job)
        entries = []
        for idx, (n, oracle, k) in enumerate(STAGE3_CONFIG_ORDER):
            counts = _counts(result[idx].data)
            assert all(len(key.replace(" ", "")) == n for key in counts), f"{jid} PUB {idx}: unexpected key width"
            e = {"n": n, "oracle": oracle, "k": k, "counts": counts}
            # Use executed-circuit resources only if the circuit is hardware-native (<= 2-qubit gates).
            if circuits is not None and max((len(i.qubits) for i in circuits[idx].data
                                             if i.operation.name not in NON_GATE_OPS), default=0) <= 2:
                e["two_qubit_gates"] = count_two_qubit_ops(circuits[idx])
                e["depth"] = circuits[idx].depth()
            entries.append(e)
        cache[bname] = {"job_id": jid, "backend": job.backend().name, "qpu_seconds": job_quantum_seconds(job),
                        "configs": entries}
        assert cache[bname]["backend"] == bname, f"{jid} ran on {cache[bname]['backend']}, not {bname}"
    STAGE3_CACHE.write_text(json.dumps(cache, indent=1))
    return cache

stage3_cache = None
if STAGE3_CACHE.exists():
    stage3_cache = json.loads(STAGE3_CACHE.read_text())
    print("Loaded cached Stage-3 counts from", STAGE3_CACHE)
elif IBM_AVAILABLE:
    stage3_cache = fetch_stage3_from_ibm()
    print("Downloaded Stage-3 counts from IBM Quantum and cached them to", STAGE3_CACHE)

recorded = {(r[0], r[1], r[2], r[3]): r for r in STAGE3_RECORDED}
stage3_rows = []
if stage3_cache is not None:
    for bname, info in stage3_cache.items():
        n_pubs = len(info["configs"])
        for e in info["configs"]:
            n = int(e["n"]); sols = solution_states(n)
            total = sum(e["counts"].values())
            psol = {s: e["counts"].get(s, 0) / total for s in sols}
            rec = recorded[(bname, n, e["oracle"], int(e["k"]))]
            stage3_rows.append({
                "backend": bname, "n": n, "oracle": e["oracle"], "k": int(e["k"]), "shots": total,
                "P_valid": sum(psol.values()), **{f"P_{s}": p for s, p in psol.items()},
                "two_qubit_gates": e.get("two_qubit_gates", rec[8]), "depth": e.get("depth", rec[9]),
                "qpu_seconds": info["qpu_seconds"] / n_pubs if np.isfinite(info["qpu_seconds"]) else np.nan,
                "job_id": info["job_id"], "scored_from": "raw_counts",
            })
else:
    print("WARNING: IBM unreachable and no cached counts; using the values recorded from the Stage-3 jobs.")
    for r in STAGE3_RECORDED:
        bname, n, oracle, k = r[:4]
        stage3_rows.append({
            "backend": bname, "n": n, "oracle": oracle, "k": k, "shots": STAGE3_SHOTS, "P_valid": r[4],
            **{f"P_{s}": p for s, p in zip(solution_states(n), r[5:8])},
            "two_qubit_gates": r[8], "depth": r[9], "qpu_seconds": np.nan,
            "job_id": STAGE3_JOB_IDS[bname], "scored_from": "recorded_output_of_job",
        })

raw_reference_df = pd.DataFrame(stage3_rows).sort_values(["backend","n","oracle","k"]).reset_index(drop=True)
raw_reference_df["method"] = "RAW"
raw_reference_df["readout_corrected"] = False
raw_reference_df["config_id"] = ("n" + raw_reference_df["n"].astype(str) + "_O" + raw_reference_df["oracle"]
                                 + "_k" + raw_reference_df["k"].astype(str))
raw_reference_df["P_ideal"] = raw_reference_df.apply(lambda r: ideal_grover_probability(int(r.n), int(r.k)), axis=1)
raw_reference_df["P_random"] = raw_reference_df["n"].map(lambda n: random_baseline(int(n)))
raw_reference_df["sigma_P"] = np.sqrt(raw_reference_df["P_valid"] * (1 - raw_reference_df["P_valid"]) / raw_reference_df["shots"])
raw_reference_df["CI95_low"] = np.maximum(0, raw_reference_df["P_valid"] - 1.96 * raw_reference_df["sigma_P"])
raw_reference_df["CI95_high"] = np.minimum(1, raw_reference_df["P_valid"] + 1.96 * raw_reference_df["sigma_P"])
raw_reference_df["relative_to_ideal"] = raw_reference_df["P_valid"] / raw_reference_df["P_ideal"]
raw_reference_df["hardware_loss"] = raw_reference_df["P_ideal"] - raw_reference_df["P_valid"]
raw_reference_df["expected_shots"] = 1 / raw_reference_df["P_valid"]
raw_reference_df["above_random_baseline"] = raw_reference_df["CI95_low"] > raw_reference_df["P_random"]

# Consistency check: re-scored counts must reproduce the originally reported values.
for _, r in raw_reference_df.iterrows():
    rec = recorded[(r.backend, int(r.n), r.oracle, int(r.k))]
    if abs(r.P_valid - rec[4]) > 1e-9:
        warnings.warn(f"{r.backend} {r.config_id}: re-scored P_valid {r.P_valid:.4f} differs from recorded {rec[4]:.4f}")

assert len(raw_reference_df) == len(BACKENDS) * len(CONFIGS), "Stage 3 must cover every backend x configuration."
raw_reference_df.to_csv(TAB_DIR / "stage3_raw_hardware_characterization.csv", index=False)
compatible_raw_df = raw_reference_df.copy()

print("Stage-3 rows:", len(raw_reference_df), "| source:", raw_reference_df.scored_from.iloc[0])
print("Rows whose 95% CI lies above random guessing (M/N):", int(raw_reference_df.above_random_baseline.sum()), "of", len(raw_reference_df))
display(raw_reference_df[["backend","n","oracle","k","P_valid","sigma_P","P_ideal","P_random","above_random_baseline",
                          "two_qubit_gates","depth","job_id","scored_from"]])

In [ ]:
# ============================================================
# 10. CURRENT TRANSPILATION RESOURCES + TRANSPILED DIAGRAMS
# ============================================================
# Fresh transpilation against today's calibration (used for new Stage-4/6 jobs).
# Stage-3 resources come from the circuits that actually ran (previous cell).
resource_rows = []
if IBM_AVAILABLE:
    for bname in BACKENDS:
        for cfg in CONFIGS:
            tq = isa_circuit(bname, cfg["n"], cfg["oracle"], cfg["k"], True)
            resource_rows.append({
                **cfg, "backend": bname,
                "two_qubit_gates": count_two_qubit_ops(tq),
                "depth": tq.depth(),
                "active_physical_qubits": active_qubits(tq),
                "data_physical_qubits": data_physical_qubits(tq, cfg["n"]),
            })
    for bname in BACKENDS:
        for n, oracle, k in REPRESENTATIVE:
            save_circuit_figure(isa_circuit(bname, n, oracle, k, True),
                                f"transpiled_{bname}_n{n}_oracle{oracle}_k{k}",
                                title=f"Transpiled Grover K-SAT circuit — {bname}, n={n}, Oracle {oracle}, k={k}",
                                fold=18, idle_wires=False)
    print("Transpiled diagrams saved to", DIAG_DIR)

resource_df = pd.DataFrame(resource_rows)
if not resource_df.empty:
    display(resource_df)
    resource_df.to_csv(TAB_DIR / "stage3_current_transpilation_resources.csv", index=False)
else:
    print("IBM not reachable: Stage-3 executed-circuit resources are used; transpiled diagrams skipped.")

## Stage 4 — Error suppression and mitigation

The four primary techniques are retained individually: **DD**, **PT**, **ZNE**, and **TREX**. `ALL4` is a sensitivity configuration only. **Every method is readout-corrected** (`READOUT_CORRECTION = True`):

| Method | Primitive | Readout correction |
|---|---|---|
| RAW (“None”), DD, PT | Sampler | tensored calibration matrix from \|0…0⟩/\|1…1⟩ circuits on the same physical qubits, in the same job |
| ZNE, ALL4 | Estimator | TREX measurement mitigation enabled together with ZNE |
| TREX | Estimator | TREX (it *is* the readout mitigation) |

- DD and PT are treated as **error suppression / randomized compilation** strategies.
- ZNE and TREX are treated as **Estimator-based expectation-value mitigation**.
- For ZNE/TREX/ALL4, `P_valid` is the expectation value of the diagonal projector onto the satisfying assignments; it is not a directly sampled histogram probability.
- Sampler rows keep both `P_valid_uncorrected` and the readout-corrected `P_valid`; the corrected uncertainty is obtained by parametric bootstrap over data and calibration counts.
- Stage-3 RAW rows are **not** readout-corrected (`readout_corrected = False`). A new readout-corrected RAW run (the “None” bar) replaces them in the active matrix when it completes.
- QPU cost is recorded two ways: measured `qpu_seconds` (job usage apportioned to PUBs by shots), and the hardware-agnostic `qpu_cost_2q_executions = shots × 2Q gates × gate-execution factor` (factor = Σ noise factors for ZNE/ALL4, 1 otherwise).

In [ ]:
# ============================================================
# 11. OBSERVABLE: PROJECTOR ONTO SATISFYING ASSIGNMENTS
# ============================================================
from qiskit.quantum_info import SparsePauliOp

def solution_projector_observable(num_qubits, n, solutions):
    """Projector P_valid = sum_s |s><s| on data qubits 0..n-1 of a LOGICAL circuit.

    `solutions` are Qiskit count keys (character n-1-q is qubit q). Build it on the
    logical circuit width and map it to the ISA circuit with .apply_layout(isa.layout).
    (v2 built it on the ISA width, so apply_layout failed silently and the projector
    was measured on physical qubits 0..n-1 instead of the circuit's data qubits.)
    """
    terms = {}
    for s in solutions:
        for mask in range(2 ** n):
            coeff = 1.0 / (2 ** n)
            chars_q = ["I"] * num_qubits
            for q in range(n):
                if (mask >> q) & 1:
                    chars_q[q] = "Z"
                    if s[n - 1 - q] == "1":
                        coeff *= -1
            label = "".join(reversed(chars_q))
            terms[label] = terms.get(label, 0.0) + coeff
    return SparsePauliOp.from_list([(label, coeff) for label, coeff in terms.items() if abs(coeff) > 1e-15])

def estimator_circuit(n, oracle, k):
    return ksat_circuit(n, oracle, k, measure=False)

# Self-test: <P_valid> on the exact statevector must equal the exact solution probability.
for cfg in CONFIGS:
    qc = estimator_circuit(cfg["n"], cfg["oracle"], cfg["k"])
    obs = solution_projector_observable(qc.num_qubits, cfg["n"], solution_states(cfg["n"]))
    ev = float(np.real(Statevector(qc).expectation_value(obs)))
    assert abs(ev - ideal_grover_probability(cfg["n"], cfg["k"])) < 1e-6, (cfg, ev)
print("Projector observable verified against exact statevector for all 8 configurations.")

In [ ]:
# ============================================================
# 12. READOUT CORRECTION (SAMPLER METHODS)
# ============================================================
def counts_to_vector(counts, n):
    v = np.zeros(2 ** n)
    for key, c in counts.items():
        v[int(key.replace(" ", ""), 2)] += c
    return v

def readout_matrices(cal0_counts, cal1_counts, n):
    """Per-qubit 2x2 assignment matrices A_q[measured, prepared] from |0..0>/|1..1> calibration."""
    v0, v1 = counts_to_vector(cal0_counts, n), counts_to_vector(cal1_counts, n)
    v0, v1 = v0 / v0.sum(), v1 / v1.sum()
    idx = np.arange(2 ** n)
    mats = []
    for q in range(n):
        bit = (idx >> q) & 1
        p1_given0 = v0[bit == 1].sum()
        p0_given1 = v1[bit == 0].sum()
        mats.append(np.array([[1 - p1_given0, p0_given1],
                              [p1_given0, 1 - p0_given1]]))
    return mats

def apply_readout_correction(prob_vec, mats):
    """Invert the tensored assignment matrix, then project to the nearest probability vector."""
    n = len(mats)
    p = prob_vec.reshape([2] * n)               # axis 0 = qubit n-1 (big-endian index)
    for q, A in enumerate(mats):
        axis = n - 1 - q
        p = np.moveaxis(np.tensordot(np.linalg.inv(A), p, axes=([1], [axis])), 0, axis)
    p = p.reshape(-1)
    p = np.clip(p, 0, None)
    return p / p.sum() if p.sum() > 0 else p

def solution_indices(n):
    return [int(s, 2) for s in solution_states(n)]

def sampler_pvalid(counts, n, cal0=None, cal1=None, bootstrap=BOOTSTRAP_SAMPLES, rng=RNG):
    """Return uncorrected and (if calibration given) readout-corrected P_valid with uncertainties."""
    v = counts_to_vector(counts, n)
    total = v.sum()
    sol = solution_indices(n)
    p_raw = v[sol].sum() / total
    out = {"shots": int(total), "P_valid_uncorrected": p_raw,
           "sigma_P_uncorrected": math.sqrt(max(p_raw * (1 - p_raw) / total, 0)),
           **{f"P_{s}": v[int(s, 2)] / total for s in solution_states(n)}}
    if cal0 is None or cal1 is None:
        out.update({"P_valid": p_raw, "sigma_P": out["sigma_P_uncorrected"],
                    "readout_corrected": False, "uncertainty_source": "binomial"})
        return out
    mats = readout_matrices(cal0, cal1, n)
    p_corr = apply_readout_correction(v / total, mats)[sol].sum()
    c0, c1 = counts_to_vector(cal0, n), counts_to_vector(cal1, n)
    boots = []
    for _ in range(bootstrap):
        vb = rng.multinomial(int(total), v / total)
        m_b = readout_matrices(
            {format(i, f"0{n}b"): c for i, c in enumerate(rng.multinomial(int(c0.sum()), c0 / c0.sum()))},
            {format(i, f"0{n}b"): c for i, c in enumerate(rng.multinomial(int(c1.sum()), c1 / c1.sum()))}, n)
        boots.append(apply_readout_correction(vb / total, m_b)[sol].sum())
    out.update({"P_valid": float(p_corr), "sigma_P": float(np.std(boots, ddof=1)),
                "readout_corrected": True, "uncertainty_source": "bootstrap_readout_corrected"})
    return out

# Self-test: a known readout channel applied to a known distribution is undone.
_n = 3
_true = RNG.dirichlet(np.ones(2 ** _n))
_mats = [np.array([[0.97, 0.05], [0.03, 0.95]]), np.array([[0.98, 0.04], [0.02, 0.96]]),
         np.array([[0.96, 0.06], [0.04, 0.94]])]
_noisy = _true.reshape([2] * _n)
for _q, _A in enumerate(_mats):
    _ax = _n - 1 - _q
    _noisy = np.moveaxis(np.tensordot(_A, _noisy, axes=([1], [_ax])), 0, _ax)
assert np.allclose(apply_readout_correction(_noisy.reshape(-1), _mats), _true, atol=1e-10)
print("Readout-correction self-test PASSED.")

In [ ]:
# ============================================================
# 13. RUNTIME PRIMITIVE FACTORIES + JOB HELPERS
# ============================================================
def _estimator_cls():
    try:
        from qiskit_ibm_runtime.executor_estimator import Estimator
    except ImportError:
        from qiskit_ibm_runtime import EstimatorV2 as Estimator
    return Estimator

def _sampler_cls():
    try:
        from qiskit_ibm_runtime.executor_sampler import Sampler
    except ImportError:
        from qiskit_ibm_runtime import SamplerV2 as Sampler
    return Sampler

def make_estimator(backend, method, shots=SHOTS):
    est = _estimator_cls()(mode=backend)
    est.options.default_shots = shots
    est.options.resilience_level = 0

    est.options.dynamical_decoupling.enable = method in {"DD", "ALL4"}
    if method in {"DD", "ALL4"}:
        est.options.dynamical_decoupling.sequence_type = "XX"

    est.options.twirling.enable_gates = method in {"PT", "ALL4"}
    trex = method == "TREX" or (READOUT_CORRECTION and method in ESTIMATOR_METHODS)
    est.options.twirling.enable_measure = trex
    est.options.twirling.num_randomizations = TWIRL_RANDOMIZATIONS

    est.options.resilience.measure_mitigation = trex
    if trex:
        est.options.resilience.measure_noise_learning.num_randomizations = TREX_RANDOMIZATIONS

    est.options.resilience.zne_mitigation = method in {"ZNE", "ALL4"}
    if method in {"ZNE", "ALL4"}:
        est.options.resilience.zne.noise_factors = ZNE_NOISE_FACTORS
        est.options.resilience.zne.extrapolator = ZNE_EXTRAPOLATOR
    return est

def make_sampler(backend, method, shots=SHOTS):
    sampler = _sampler_cls()(mode=backend)
    sampler.options.default_shots = shots
    sampler.options.dynamical_decoupling.enable = method in {"DD", "ALL4"}
    if method in {"DD", "ALL4"}:
        sampler.options.dynamical_decoupling.sequence_type = "XX"
    sampler.options.twirling.enable_gates = method in {"PT", "ALL4"}
    sampler.options.twirling.num_randomizations = TWIRL_RANDOMIZATIONS
    return sampler

def gate_execution_factor(method):
    """2Q-gate executions per shot relative to the unmitigated circuit."""
    return float(sum(ZNE_NOISE_FACTORS)) if method in {"ZNE", "ALL4"} else 1.0

def calibration_circuits(bname, n, phys):
    from qiskit import transpile
    out = []
    for prep in (0, 1):
        qc = QuantumCircuit(QuantumRegister(n, "q"), ClassicalRegister(n, "meas"))
        if prep:
            qc.x(range(n))
        qc.measure(range(n), range(n))
        out.append(transpile(qc, backend_objects[bname], initial_layout=phys,
                             optimization_level=1, seed_transpiler=SEED))
    return out

def _job_paths(bname, method, prefix):
    stem = f"{bname}_{prefix}{method}"
    return JOB_DIR / f"{stem}_job_id.txt", JOB_DIR / f"{stem}_manifest.json"

def pub_layout(method, cfgs):
    """Deterministic PUB index layout of a job (identical at submission and retrieval)."""
    meta, i = [], 0
    for cfg in cfgs:
        e = {"n": int(cfg["n"]), "oracle": str(cfg["oracle"]), "k": int(cfg["k"]), "pub_index": i}
        i += 1
        if method in SUPPRESSION_METHODS and READOUT_CORRECTION:
            e["cal0_index"], e["cal1_index"] = i, i + 1
            i += 2
        meta.append(e)
    return meta

def submit_job(bname, method, cfgs, shots, prefix=""):
    """Submit one job for `method` on the real processor `bname` covering every config in `cfgs`.

    Sampler jobs carry their own readout-calibration circuits. The manifest records,
    per config, the PUB indices plus the transpiled 2Q count and depth that ran.
    """
    backend = backend_objects[bname]
    meta = pub_layout(method, cfgs)
    pubs = []
    for e in meta:
        n, oracle, k = e["n"], e["oracle"], e["k"]
        if method in SUPPRESSION_METHODS:
            tq = isa_circuit(bname, n, oracle, k, True)
            pubs.append((tq, None, shots))
            phys = data_physical_qubits(tq, n)
            e["data_physical_qubits"] = phys
            if READOUT_CORRECTION:
                c0, c1 = calibration_circuits(bname, n, phys)
                pubs.extend([(c0, None, CAL_SHOTS), (c1, None, CAL_SHOTS)])
        else:
            logical = estimator_circuit(n, oracle, k)
            tq = isa_circuit(bname, n, oracle, k, False)
            obs = solution_projector_observable(logical.num_qubits, n, solution_states(n)).apply_layout(tq.layout)
            pubs.append((tq, obs))
        e["two_qubit_gates"] = count_two_qubit_ops(tq)
        e["depth"] = tq.depth()

    factory = make_sampler if method in SUPPRESSION_METHODS else make_estimator
    job = factory(backend, method, shots).run(pubs)

    job_file, manifest_file = _job_paths(bname, method, prefix)
    job_file.write_text(str(job.job_id()))
    manifest_file.write_text(json.dumps({"method": method, "shots": shots, "cal_shots": CAL_SHOTS,
                                         "readout_correction": READOUT_CORRECTION,
                                         "configs": meta}, indent=2))
    print(f"{prefix}{method} on {bname}: {len(pubs)} PUBs -> job {job.job_id()}   (save this ID)")
    return job.job_id()

def _scalar(x):
    return float(np.asarray(x).reshape(-1)[0])

def retrieve_job(bname, method, cfgs, shots, prefix=""):
    """Rows for a completed IBM Quantum job; [] if not submitted or not finished.

    The job ID comes from JOB_ID_OVERRIDES (for jobs submitted in another session or on
    another machine) or from the job-ID file written at submission. If the manifest file
    is missing, the PUB layout is rebuilt deterministically from `cfgs`.
    """
    job_file, manifest_file = _job_paths(bname, method, prefix)
    job_id = JOB_ID_OVERRIDES.get(f"{prefix}{bname}/{method}")
    if job_id is None and job_file.exists():
        job_id = job_file.read_text().strip()
    if job_id is None:
        return []
    if manifest_file.exists():
        manifest = json.loads(manifest_file.read_text())
    else:
        manifest = {"method": method, "shots": shots, "cal_shots": CAL_SHOTS,
                    "readout_correction": READOUT_CORRECTION, "configs": pub_layout(method, cfgs)}
    job = service.job(job_id)
    status = str(job.status()).upper()
    print(f"{bname}/{prefix}{method}: {job_id} -> {status}")
    if not any(s in status for s in ("DONE", "COMPLETED")):
        return []
    assert job.backend().name == bname, f"Job {job_id} ran on {job.backend().name}, not {bname}"
    result = job.result()
    expected_pubs = max(max(e.get("cal1_index", e["pub_index"]), e["pub_index"]) for e in manifest["configs"]) + 1
    if len(result) != expected_pubs:
        raise RuntimeError(f"Job {job_id} has {len(result)} PUBs but {bname}/{prefix}{method} expects "
                           f"{expected_pubs}. Check that the job ID belongs to this processor/strategy/stage.")
    qpu_s = job_quantum_seconds(job)
    executed = _executed_circuits(job)

    # Apportion measured QPU time to PUBs by shots.
    weights = []
    for e in manifest["configs"]:
        w = manifest["shots"] * gate_execution_factor(method)
        if "cal0_index" in e:
            w += 2 * manifest["cal_shots"]
        weights.append(w)
    weights = np.asarray(weights) / np.sum(weights)

    rows = []
    for e, w in zip(manifest["configs"], weights):
        n = int(e["n"])
        pub = result[e["pub_index"]]
        if method in SUPPRESSION_METHODS:
            cal0 = _counts(result[e["cal0_index"]].data) if "cal0_index" in e else None
            cal1 = _counts(result[e["cal1_index"]].data) if "cal1_index" in e else None
            rec = sampler_pvalid(_counts(pub.data), n, cal0, cal1)
            rec["shot_overhead_factor"] = 1.0
        else:
            pv = _scalar(pub.data.evs)
            try:
                sg = _scalar(pub.data.stds)
            except Exception:
                sg = np.nan
            rec = {"shots": manifest["shots"], "P_valid": pv, "sigma_P": sg,
                   "P_valid_uncorrected": np.nan, "readout_corrected": READOUT_CORRECTION or method == "TREX",
                   "uncertainty_source": "estimator_stds",
                   "shot_overhead_factor": float(len(ZNE_NOISE_FACTORS)) if method in {"ZNE", "ALL4"} else 1.0}
        g2q, depth = e.get("two_qubit_gates"), e.get("depth")
        if g2q is None and executed is not None and max(
                (len(i.qubits) for i in executed[e["pub_index"]].data if i.operation.name not in NON_GATE_OPS), default=0) <= 2:
            g2q, depth = count_two_qubit_ops(executed[e["pub_index"]]), executed[e["pub_index"]].depth()
        sg = rec["sigma_P"]
        rows.append({
            "backend": bname, "n": n, "oracle": e["oracle"], "k": int(e["k"]), "method": method,
            "config_id": f"n{n}_O{e['oracle']}_k{int(e['k'])}",
            **rec, "nominal_shots": manifest["shots"],
            "CI95_low": max(0, rec["P_valid"] - 1.96 * sg) if np.isfinite(sg) else np.nan,
            "CI95_high": min(1, rec["P_valid"] + 1.96 * sg) if np.isfinite(sg) else np.nan,
            "two_qubit_gates": g2q, "depth": depth,
            "qpu_seconds": qpu_s * w if np.isfinite(qpu_s) else np.nan,
            "job_id": job_id, "status": "complete", "data_source": "new_qpu",
        })
    return rows

print("Runtime factories and job helpers defined.")

In [ ]:
# ============================================================
# 14. HARDWARE CAMPAIGN MANIFEST
# ============================================================
campaign_rows = []
for bname in BACKENDS:
    for cfg in CONFIGS:
        for method in METHODS:
            campaign_rows.append({
                "backend": bname, **cfg,
                "config_id": f"n{cfg['n']}_O{cfg['oracle']}_k{cfg['k']}",
                "method": method, "status": "pending", "shots": SHOTS,
                "data_source": None, "readout_corrected": None,
            })
campaign_df = pd.DataFrame(campaign_rows)

# Stage-3 raw characterization fills the RAW cells until a new readout-corrected RAW run completes.
if USE_STAGE3_AS_RAW_BASELINE:
    for _, r in compatible_raw_df.iterrows():
        mask = ((campaign_df.backend == r.backend) & (campaign_df.n == r.n) &
                (campaign_df.oracle == r.oracle) & (campaign_df.k == r.k) &
                (campaign_df.method == "RAW"))
        campaign_df.loc[mask, "status"] = "reference_complete"
        campaign_df.loc[mask, "data_source"] = "stage3_raw_characterization"
        campaign_df.loc[mask, "readout_corrected"] = False
        campaign_df.loc[mask, "uncertainty_source"] = "binomial"
        for col in ["P_valid", "sigma_P", "CI95_low", "CI95_high", "expected_shots"]:
            campaign_df.loc[mask, col] = r[col]

print("Total campaign cells:", len(campaign_df))
print(campaign_df.groupby(["backend","method"]).size().unstack())
campaign_df.to_csv(TAB_DIR / "stage4_hardware_campaign_manifest.csv", index=False)

### QPU submission strategy

The primary campaign contains **2 processors × 6 methods × 8 configurations = 96 cells** (one job per processor × method). `ALL4` remains sensitivity analysis.

For RAW/DD/PT, sampled probabilities are obtained with Sampler, with readout-calibration circuits in the same job. For ZNE/TREX/ALL4, the satisfying-state projector is evaluated with Estimator. All primary-campaign cells use the same nominal 500-shot budget.

In [ ]:
# ============================================================
# 15. SUBMIT PRIMARY QPU CAMPAIGN — DISABLED BY DEFAULT
# ============================================================
job_manifest = []

if SUBMIT_STAGE4_JOBS:
    if not IBM_AVAILABLE:
        raise RuntimeError("IBM Quantum processors are not available.")
    for bname in BACKENDS:
        for method in STAGE4_METHODS:
            jid = submit_job(bname, method, CONFIGS, SHOTS)
            job_manifest.append({"backend": bname, "method": method, "job_id": jid,
                                 "n_configs": len(CONFIGS), "nominal_shots": SHOTS})
    pd.DataFrame(job_manifest).to_csv(JOB_DIR / "submission_manifest.csv", index=False)
else:
    print("Stage-4 submission skipped. Set SUBMIT_STAGE4_JOBS=True only after reviewing the manifest.")

In [ ]:
# ============================================================
# 16. RETRIEVE PRIMARY QPU JOBS
# ============================================================
stage4_rows = []
if IBM_AVAILABLE:
    for bname in BACKENDS:
        for method in METHODS:
            stage4_rows.extend(retrieve_job(bname, method, CONFIGS, SHOTS))

stage4_new_df = pd.DataFrame(stage4_rows)
if not stage4_new_df.empty:
    stage4_new_df.to_csv(TAB_DIR / "stage4_completed_primary_campaign.csv", index=False)
    display(stage4_new_df)
else:
    print("No new primary-campaign jobs have completed yet.")

In [ ]:
# ============================================================
# 17. MASTER RESULT MATRIX — NO FABRICATION, NO INCOMPATIBLE RAW
# ============================================================
KEYS = ["backend", "n", "oracle", "k", "method"]
master_df = campaign_df.set_index(KEYS)

# New QPU results take precedence over Stage-3 values for the same cell.
if not stage4_new_df.empty:
    new = stage4_new_df.drop_duplicates(KEYS, keep="last").set_index(KEYS)
    for col in new.columns:
        if col not in master_df.columns:
            master_df[col] = np.nan
        master_df[col] = master_df[col].astype(object)
        hit = master_df.index.intersection(new.index)
        master_df.loc[hit, col] = new.loc[hit, col]
master_df = master_df.reset_index()
for col in ["P_valid", "sigma_P", "two_qubit_gates", "depth", "qpu_seconds",
            "shot_overhead_factor", "P_valid_uncorrected", "shots"]:
    if col not in master_df.columns:
        master_df[col] = np.nan
    master_df[col] = pd.to_numeric(master_df[col], errors="coerce")

# Resources: (1) circuit that actually ran, (2) Stage-3 executed circuits, (3) current transpilation.
master_df["resource_source"] = np.where(master_df["two_qubit_gates"].notna(), "executed_circuit", None)
fills = [("stage3_executed_circuit", raw_reference_df[["backend","n","oracle","k","two_qubit_gates","depth"]])]
if not resource_df.empty:
    fills.append(("current_transpilation", resource_df[["backend","n","oracle","k","two_qubit_gates","depth"]]))
for source, ref in fills:
    m = master_df.merge(ref.drop_duplicates(["backend","n","oracle","k"]),
                        on=["backend","n","oracle","k"], how="left", suffixes=("", "_ref"))
    missing = master_df["two_qubit_gates"].isna() & m["two_qubit_gates_ref"].notna()
    master_df.loc[missing, "two_qubit_gates"] = m.loc[missing, "two_qubit_gates_ref"]
    master_df.loc[missing, "depth"] = m.loc[missing, "depth_ref"]
    master_df.loc[missing, "resource_source"] = source

master_df["P_ideal"] = master_df.apply(lambda r: ideal_grover_probability(int(r.n), int(r.k)), axis=1)
master_df["P_random"] = master_df["n"].map(lambda n: random_baseline(int(n)))
master_df["relative_to_ideal"] = master_df["P_valid"] / master_df["P_ideal"]
master_df["hardware_loss"] = master_df["P_ideal"] - master_df["P_valid"]
master_df["expected_shots"] = np.where(master_df["P_valid"] > 0, 1.0 / master_df["P_valid"], np.inf)

# Do not infer uncertainty for Estimator values from a binomial model. Only
# sampled RAW/DD/PT cells may use the binomial fallback when necessary.
fallback = master_df["sigma_P"].isna() & master_df["method"].isin(SUPPRESSION_METHODS) & master_df["P_valid"].notna()
master_df.loc[fallback, "sigma_P"] = np.sqrt(master_df.loc[fallback, "P_valid"] * (1 - master_df.loc[fallback, "P_valid"]) / master_df.loc[fallback, "shots"])
master_df.loc[fallback, "uncertainty_source"] = "binomial"

master_df["CI95_low"] = np.where(master_df["sigma_P"].notna(), np.maximum(0, master_df["P_valid"] - 1.96 * master_df["sigma_P"]), np.nan)
master_df["CI95_high"] = np.where(master_df["sigma_P"].notna(), np.minimum(1, master_df["P_valid"] + 1.96 * master_df["sigma_P"]), np.nan)
master_df["above_random_baseline"] = master_df["CI95_low"] > master_df["P_random"]

# QPU cost and time-to-solution.
master_df["gate_execution_factor"] = master_df["method"].map(gate_execution_factor)
master_df["qpu_cost_2q_executions"] = master_df["shots"] * master_df["two_qubit_gates"] * master_df["gate_execution_factor"]
master_df["expected_2q_executions_to_solution"] = master_df["expected_shots"] * master_df["two_qubit_gates"] * master_df["gate_execution_factor"]
master_df["qpu_seconds_per_shot"] = master_df["qpu_seconds"] / master_df["shots"]
master_df["time_to_solution_s"] = master_df["expected_shots"] * master_df["qpu_seconds_per_shot"]

master_df.to_csv(TAB_DIR / "stage4_master_result_matrix.csv", index=False)
display(master_df.head(24))

## Stage 5 — HAGCS evaluation

HAGCS is treated here as the paper's **proposed selection framework**, not as a pre-existing standard metric. It runs in two steps, matching the pipeline figure:

1. **Pareto-based feasibility filtering.** Per (processor, n), a configuration–method pair is *eligible* if `P_valid`, `sigma_P`, two-qubit count and depth are all known. Among eligible pairs, only those **not Pareto-dominated** on the objectives in `HAGCS_PARETO_OBJECTIVES` (maximize \(P_{valid}-2\sigma_P\); minimize 2Q gates, depth, and QPU cost) are feasible. With `REQUIRE_ABOVE_RANDOM = True`, a pair must also have its 95% CI above the random-guessing baseline M/N.
2. **Uncertainty-aware, cost-sensitive ranking** of the feasible pairs by

\[J_{robust}=\frac{\max(0,P_{valid}-2\sigma_P)}{G_{2Q}},\]

with ties broken by `P_valid`. Selection is over the primary methods (RAW, DD, PT, ZNE, TREX); ALL4 is sensitivity analysis only and is never selected.

The **selection trade-off** table compares each HAGCS pick with the max-`P_valid` pick on the same primary data (fewer 2Q gates vs ΔP_valid). It is not an independent validation; Stage 6 provides that.

In [ ]:
# ============================================================
# 18. HAGCS: PARETO FEASIBILITY FILTER -> ROBUST RANKING
# ============================================================
HAGCS_METHODS = PRIMARY_METHODS

hagcs_df = master_df.copy()
hagcs_df["P_lower_2sigma"] = np.maximum(0, hagcs_df["P_valid"] - 2 * hagcs_df["sigma_P"])
hagcs_df["eligible"] = (hagcs_df[["P_valid", "sigma_P", "two_qubit_gates", "depth"]].notna().all(axis=1)
                        & (hagcs_df["two_qubit_gates"] > 0)
                        & hagcs_df["method"].isin(HAGCS_METHODS))
hagcs_df["J_robust"] = np.where(hagcs_df["eligible"], hagcs_df["P_lower_2sigma"] / hagcs_df["two_qubit_gates"], np.nan)
hagcs_df["P_valid_per_2Q_gate"] = hagcs_df["P_valid"] / hagcs_df["two_qubit_gates"]
hagcs_df["P_valid_per_depth"] = hagcs_df["P_valid"] / hagcs_df["depth"]

# Mitigation gain against RAW on the same backend/configuration.
raw_lookup = (hagcs_df[hagcs_df["method"] == "RAW"][["backend","n","oracle","k","P_valid","readout_corrected"]]
              .rename(columns={"P_valid": "P_raw", "readout_corrected": "raw_readout_corrected"}))
hagcs_df = hagcs_df.merge(raw_lookup, on=["backend","n","oracle","k"], how="left")
hagcs_df["absolute_mitigation_gain"] = hagcs_df["P_valid"] - hagcs_df["P_raw"]
hagcs_df["relative_mitigation_gain_pct"] = np.where(hagcs_df["P_raw"] > 0, 100 * hagcs_df["absolute_mitigation_gain"] / hagcs_df["P_raw"], np.nan)
# A gain is like-for-like only if RAW and the method are both readout-corrected (or both not).
hagcs_df["gain_like_for_like"] = hagcs_df["readout_corrected"].astype(object).eq(hagcs_df["raw_readout_corrected"].astype(object))

def pareto_mask(df, objectives):
    """True for rows not dominated on `objectives` ({column: 'max'|'min'})."""
    cols = [c for c in objectives if c in df.columns and df[c].notna().all()]
    if df.empty or not cols:
        return pd.Series(False, index=df.index)
    vals = np.column_stack([df[c].to_numpy(float) * (-1.0 if objectives[c] == "max" else 1.0) for c in cols])
    keep = []
    for i in range(len(vals)):
        dominated = np.any(np.all(vals <= vals[i], axis=1) & np.any(vals < vals[i], axis=1))
        keep.append(not dominated)
    return pd.Series(keep, index=df.index)

hagcs_df["feasible"] = hagcs_df["eligible"] & (hagcs_df["above_random_baseline"].fillna(False).astype(bool)
                                               if REQUIRE_ABOVE_RANDOM else True)
hagcs_df["pareto_optimal"] = False
for _, g in hagcs_df[hagcs_df["feasible"]].groupby(["backend", "n"]):
    hagcs_df.loc[g.index, "pareto_optimal"] = pareto_mask(g, HAGCS_PARETO_OBJECTIVES)

hagcs_df["rank_P_valid"] = hagcs_df[hagcs_df["eligible"]].groupby(["backend","n"])["P_valid"].rank(ascending=False, method="min")
hagcs_df["rank_J_robust"] = hagcs_df[hagcs_df["pareto_optimal"]].groupby(["backend","n"])["J_robust"].rank(ascending=False, method="min")
hagcs_df.to_csv(TAB_DIR / "stage5_HAGCS_complete_metrics.csv", index=False)

# Step 2: rank Pareto-feasible pairs; one selection per (backend, n).
selected_hagcs = (
    hagcs_df[hagcs_df["pareto_optimal"]]
    .sort_values(["backend","n","J_robust","P_valid"], ascending=[True,True,False,False])
    .groupby(["backend","n"], as_index=False).head(1).reset_index(drop=True)
)
selected_hagcs.to_csv(TAB_DIR / "stage5_HAGCS_selected_for_validation.csv", index=False)

# Comparison point: highest P_valid among eligible pairs (P_valid alone).
max_pvalid = (
    hagcs_df[hagcs_df["eligible"]]
    .sort_values(["backend","n","P_valid","two_qubit_gates"], ascending=[True,True,False,True])
    .groupby(["backend","n"], as_index=False).head(1).reset_index(drop=True)
)

tradeoff_rows = []
for _, s in selected_hagcs.iterrows():
    m = max_pvalid[(max_pvalid.backend == s.backend) & (max_pvalid.n == s.n)].iloc[0]
    dP = s.P_valid - m.P_valid
    sigma_dP = math.sqrt(s.sigma_P ** 2 + m.sigma_P ** 2)
    tradeoff_rows.append({
        "backend": s.backend, "n": int(s.n),
        "HAGCS_selected": f"O{s.oracle}/k={int(s.k)}/{s.method}",
        "max_P_valid_choice": f"O{m.oracle}/k={int(m.k)}/{m.method}",
        "P_valid_HAGCS": s.P_valid, "P_valid_maxP": m.P_valid,
        "delta_P_valid": dP, "sigma_delta_P": sigma_dP,
        "delta_within_2sigma": abs(dP) <= 2 * sigma_dP,
        "2Q_HAGCS": int(s.two_qubit_gates), "2Q_maxP": int(m.two_qubit_gates),
        "2Q_reduction_pct": 100 * (1 - s.two_qubit_gates / m.two_qubit_gates),
        "depth_reduction_pct": 100 * (1 - s.depth / m.depth),
        "HAGCS_above_random_baseline": bool(s.above_random_baseline),
        "same_choice": (s.oracle, s.k, s.method) == (m.oracle, m.k, m.method),
        "data_source": s.data_source,
    })
tradeoff_df = pd.DataFrame(tradeoff_rows)
tradeoff_df.to_csv(TAB_DIR / "stage5_HAGCS_vs_maxP_tradeoff.csv", index=False)

print("Selected HAGCS configurations for Stage 6:")
display(selected_hagcs[["backend","n","oracle","k","method","P_valid","sigma_P","P_ideal","P_random",
                        "above_random_baseline","two_qubit_gates","depth","J_robust","data_source"]])
print("Selection trade-off vs max-P_valid (primary data; NOT an independent validation):")
display(tradeoff_df)
if (~selected_hagcs["above_random_baseline"].fillna(False).astype(bool)).any():
    warnings.warn("At least one HAGCS selection is not distinguishable from random guessing (95% CI includes M/N). "
                  "Report it as such, or rerun with REQUIRE_ABOVE_RANDOM=True.")

## Stage 5 — Publication tables

The exports are split so the main TQE manuscript can use compact tables while the supplementary material can contain the complete 96-record matrix.

In [ ]:
# ============================================================
# 19. PUBLICATION TABLE EXPORTER
# ============================================================
def save_table(df, stem, columns=None, float_digits=4):
    out = df.copy()
    if columns:
        out = out[[c for c in columns if c in out.columns]]
    csv_path = TAB_DIR / f"{stem}.csv"
    tex_path = TAB_DIR / f"{stem}.tex"
    out.to_csv(csv_path, index=False)
    try:
        latex = out.to_latex(index=False, escape=False, float_format=lambda x: f"{x:.{float_digits}f}" if isinstance(x,(float,np.floating)) else str(x))
        tex_path.write_text(latex, encoding="utf-8")
    except Exception as exc:
        print("LaTeX export warning:", exc)
    return csv_path, tex_path

save_table(stage1_df, "TABLE1_configuration_and_resources", ["n","N","oracle","k","P_ideal","P_random","logical_qubits","logical_2q_gates","logical_depth"])
save_table(raw_reference_df, "TABLE2_stage3_raw_hardware_characterization", ["backend","n","oracle","k","shots","P_valid","P_ideal","P_random","relative_to_ideal","sigma_P","CI95_low","CI95_high","above_random_baseline","two_qubit_gates","depth","job_id","scored_from"])
# v2 exported TABLE3 from master_df, which lacks the HAGCS columns; use hagcs_df.
save_table(hagcs_df, "TABLE3_primary_HAGCS_matrix", ["backend","n","oracle","k","method","data_source","readout_corrected","shots","P_valid","P_valid_uncorrected","P_ideal","relative_to_ideal","sigma_P","CI95_low","CI95_high","above_random_baseline","two_qubit_gates","depth","qpu_cost_2q_executions","qpu_seconds","time_to_solution_s","expected_shots","absolute_mitigation_gain","relative_mitigation_gain_pct","J_robust","pareto_optimal"])
save_table(selected_hagcs, "TABLE4_selected_HAGCS_configurations", ["backend","n","oracle","k","method","P_valid","sigma_P","P_ideal","relative_to_ideal","above_random_baseline","two_qubit_gates","depth","qpu_cost_2q_executions","expected_shots","J_robust","pareto_optimal","data_source"])
save_table(tradeoff_df, "TABLE5_HAGCS_vs_maxP_tradeoff")

print("Publication tables written to:", TAB_DIR)

## Stage 6 — Independent validation of the HAGCS-selected configuration

Stage 6 is deliberately **not** a second complete raw matrix. It validates the actual Stage-5 selection: one HAGCS-selected configuration per `(backend, n)`, including its selected method, with fresh jobs that are never fed back into selection. Until these jobs complete, no Stage-6 number may be reported — in particular, the Stage-5 trade-off table is not a substitute.

In [ ]:
# ============================================================
# 20. STAGE-6 VALIDATION MANIFEST + OPTIONAL SUBMISSION
# ============================================================
validation_configs = selected_hagcs[["backend","n","oracle","k","method"]].copy()
validation_configs["config_id"] = [f"n{int(r.n)}_O{r.oracle}_k{int(r.k)}" for r in validation_configs.itertuples()]
validation_configs["shots"] = INDEPENDENT_SHOTS
validation_configs["status"] = "pending"
validation_configs.to_csv(TAB_DIR / "stage6_independent_validation_manifest.csv", index=False)
display(validation_configs)

stage4_complete = bool(master_df.loc[master_df.method.isin(set(PRIMARY_METHODS) - {"RAW"}), "P_valid"].notna().all())
if SUBMIT_STAGE6_JOBS and not validation_configs.empty:
    if not IBM_AVAILABLE:
        raise RuntimeError("IBM Quantum processors are not available.")
    if not stage4_complete:
        warnings.warn("Stage 4 is incomplete: the selection is made from the strategies measured so far. "
                      "Validate only after the Stage-4 campaign has finished if the paper claims a selection over all strategies.")
    for (bname, method), g in validation_configs.groupby(["backend", "method"]):
        submit_job(bname, method, g[["n","oracle","k"]].to_dict("records"), INDEPENDENT_SHOTS, prefix="stage6_")
else:
    print("Stage-6 submission skipped. Set SUBMIT_STAGE6_JOBS=True after Stage 4 is complete.")

In [ ]:
# ============================================================
# 21. RETRIEVE STAGE-6 VALIDATION RESULTS
# ============================================================
indep_rows = []
if IBM_AVAILABLE and not validation_configs.empty:
    for (bname, method), g in validation_configs.groupby(["backend", "method"]):
        indep_rows.extend(retrieve_job(bname, method, g[["n","oracle","k"]].to_dict("records"),
                                       INDEPENDENT_SHOTS, prefix="stage6_"))
    # Guard against circular validation: Stage-6 jobs must differ from the jobs used for selection.
    used_for_selection = set(master_df.get("job_id", pd.Series(dtype=object)).dropna()) | set(STAGE3_JOB_IDS.values())
    assert not ({r["job_id"] for r in indep_rows} & used_for_selection), "Stage-6 reused a selection job."


independent_df = pd.DataFrame(indep_rows)
compare = pd.DataFrame()
if not independent_df.empty:
    compare = independent_df.merge(
        selected_hagcs[["backend","n","oracle","k","method","P_valid","sigma_P"]],
        on=["backend","n","oracle","k","method"], how="inner", suffixes=("_validation","_selection"))
    compare["absolute_deviation"] = compare["P_valid_validation"] - compare["P_valid_selection"]
    compare["relative_deviation_pct"] = 100 * compare["absolute_deviation"] / compare["P_valid_selection"]
    compare["sigma_deviation"] = np.sqrt(compare["sigma_P_validation"]**2 + compare["sigma_P_selection"]**2)
    compare["consistent_within_2sigma"] = compare["absolute_deviation"].abs() <= 2 * compare["sigma_deviation"]
    compare.to_csv(TAB_DIR / "stage6_independent_validation_results.csv", index=False)
    display(compare)
else:
    print("No Stage-6 validation results available yet — Stage 6 must not be reported.")

# Publication-quality figures

All figures are saved at **600 dpi** as PNG and also as vector PDF/SVG. The visual style is intentionally restrained for IEEE/TQE-style manuscripts: white background, serif typography, compact legends, explicit units, and no decorative effects. Panels with pending data say so on the figure rather than silently omitting it.

In [ ]:
# ============================================================
# 22. FIGURE HELPERS
# ============================================================
BACKEND_COLORS = {"ibm_kingston": "#1f4e9a", "ibm_marrakesh": "#c0504d"}
N_COLORS = {5: "#1f4e9a", 6: "#c0504d"}

def save_fig(stem):
    png = FIG_DIR / f"{stem}.png"
    pdf = FIG_DIR / f"{stem}.pdf"
    svg = FIG_DIR / f"{stem}.svg"
    plt.savefig(png, dpi=600, bbox_inches="tight", facecolor="white")
    plt.savefig(pdf, bbox_inches="tight", facecolor="white")
    plt.savefig(svg, bbox_inches="tight", facecolor="white")
    return png, pdf, svg

def short(b):
    return b.replace("ibm_", "")

In [ ]:
# ============================================================
# 23. FIGURE 1 — IDEAL GROVER CURVES FOR n=5 AND n=6
# ============================================================
fig, ax = plt.subplots(figsize=(7.2, 4.8))
for n in N_VALUES:
    rs = np.arange(0, 9)
    rr = np.linspace(0, 8, 400)
    theta = np.arcsin(np.sqrt(M_SOLUTIONS / 2**n))
    ax.plot(rr, np.sin((2 * rr + 1) * theta) ** 2, color=N_COLORS[n], linewidth=1.6)
    ax.plot(rs, [ideal_grover_probability(n, int(r)) for r in rs], "o", color=N_COLORS[n],
            label=f"n={n} (N={2**n})", markersize=5)
    ax.axhline(random_baseline(n), color=N_COLORS[n], linestyle=":", linewidth=1)
for k in K_VALUES:
    ax.axvline(k, color="0.6", linestyle="--", linewidth=0.8)
ax.text(K_VALUES[0] + 0.05, 1.0, "study k", fontsize=8, color="0.4", va="top")
ax.set_xlabel("Grover iterations, $r$")
ax.set_ylabel("Ideal $P_{valid}$")
ax.set_title("Ideal Grover Success Probability (dotted: random guessing M/N)")
ax.set_xticks(range(0, 9))
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.25)
ax.legend(frameon=False)
plt.tight_layout()
save_fig("FIG1_ideal_grover_probability")
plt.show()

In [ ]:
# ============================================================
# 24. FIGURE 2 — RAW HARDWARE VS IDEAL
# ============================================================
plot_df = raw_reference_df.copy()

fig, ax = plt.subplots(figsize=(10, 5.4))
x = np.arange(len(plot_df))
width = 0.34

ax.bar(x - width/2, plot_df["P_ideal"], width, label="Ideal", color="0.75")
colors = [BACKEND_COLORS[b] for b in plot_df["backend"]]
ax.bar(x + width/2, plot_df["P_valid"], width, yerr=1.96 * plot_df["sigma_P"],
       capsize=2, label="Hardware raw (95% CI)", color=colors)
ax.scatter(x, plot_df["P_random"], marker="_", s=200, color="k", label="Random guessing M/N", zorder=3)

ax.set_xticks(x)
ax.set_xticklabels([f"{short(r.backend)}\nn={r.n}, O{r.oracle}, k={r.k}" for _, r in plot_df.iterrows()],
                   rotation=70, ha="right")
ax.set_ylabel("$P_{valid}$")
ax.set_title("Raw Hardware Performance Relative to the Ideal Reference")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)
plt.tight_layout()
save_fig("FIG2_raw_hardware_vs_ideal")
plt.show()

In [ ]:
# ============================================================
# 25. FIGURE 3 — MITIGATION COMPARISON BY PROCESSOR
# ============================================================
# Pipeline panel 4: P_valid per error-management strategy, one bar per processor.
# Mean over the configurations completed for that (processor, n, method);
# the count is printed on each bar. Pending methods are labelled, not hidden.
fig, axes = plt.subplots(1, len(N_VALUES), figsize=(11, 4.6), sharey=False)
plot_methods = PRIMARY_METHODS
width = 0.38
for ax, n in zip(np.atleast_1d(axes), N_VALUES):
    x = np.arange(len(plot_methods))
    for j, bname in enumerate(BACKENDS):
        means, errs, counts = [], [], []
        for method in plot_methods:
            sub = hagcs_df[(hagcs_df.backend == bname) & (hagcs_df.n == n) &
                           (hagcs_df.method == method) & hagcs_df.P_valid.notna()]
            counts.append(len(sub))
            means.append(sub.P_valid.mean() if len(sub) else np.nan)
            errs.append(np.sqrt((sub.sigma_P ** 2).sum()) / len(sub) if len(sub) else np.nan)
        off = (j - 0.5) * width
        ax.bar(x + off, np.nan_to_num(means), width, yerr=np.nan_to_num(errs), capsize=2,
               color=BACKEND_COLORS[bname], label=short(bname))
        for xi, m, e, c in zip(x + off, means, errs, counts):
            top = (m + e) if np.isfinite(m) else 0
            ax.text(xi, top, f"n_cfg={c}" if c else "pending", ha="center", va="bottom",
                    fontsize=6.5, rotation=90)
    ax.axhline(random_baseline(n), color="k", linestyle=":", linewidth=1, label="random M/N")
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_LABELS[m] for m in plot_methods])
    ax.set_xlabel("Error-management strategy $m$")
    ax.set_title(f"n={n}")
    ax.grid(axis="y", alpha=0.25)
np.atleast_1d(axes)[0].set_ylabel("Mean $P_{valid}$ (readout-corrected where available)")
np.atleast_1d(axes)[0].legend(frameon=False)
fig.suptitle("Effect of Error Suppression and Mitigation by Processor", y=1.02)
plt.tight_layout()
save_fig("FIG3_mitigation_comparison")
plt.show()
n_done = int(hagcs_df[hagcs_df.method.isin(set(PRIMARY_METHODS) - {"RAW"})].P_valid.notna().sum())
if n_done == 0:
    print("NOTE: no DD/PT/ZNE/TREX results yet — FIG3 shows RAW only. Mitigation claims cannot be made.")

In [ ]:
# ============================================================
# 26. FIGURE 4 — P_valid VS TWO-QUBIT GATE COUNT
# ============================================================
available = hagcs_df.dropna(subset=["P_valid", "two_qubit_gates"]).copy()
if not available.empty:
    fig, ax = plt.subplots(figsize=(7.5, 5.2))
    markers = {"ibm_kingston": "o", "ibm_marrakesh": "s"}
    for method in [m for m in METHODS if m in available.method.unique()]:
        for bname in BACKENDS:
            sub = available[(available.method == method) & (available.backend == bname)]
            if sub.empty: continue
            ax.errorbar(sub["two_qubit_gates"], sub["P_valid"], yerr=1.96 * sub["sigma_P"],
                        fmt=markers[bname], ms=6, capsize=2, label=f"{METHOD_LABELS[method]} / {short(bname)}")
    ax.set_xlabel("Two-qubit gate count (transpiled)")
    ax.set_ylabel("$P_{valid}$")
    ax.set_title("Solution Probability Versus Circuit Complexity")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, ncols=2, fontsize=8)
    plt.tight_layout()
    save_fig("FIG4_Pvalid_vs_2Q_gates")
    plt.show()
else:
    print("Complexity figure pending: no resource-matched results.")

In [ ]:
# ============================================================
# 27. FIGURE 5 — FULL P_valid HEATMAP
# ============================================================
heat = (
    hagcs_df.assign(config=lambda d: "n" + d.n.astype(str) + "-O" + d.oracle + "-k" + d.k.astype(str))
    .pivot_table(index=["backend", "method"], columns="config", values="P_valid", aggfunc="mean", dropna=False)
    .reindex([(b, m) for b in BACKENDS for m in METHODS])
)
if heat.notna().any().any():
    fig, ax = plt.subplots(figsize=(12, 6))
    cmap = plt.get_cmap("viridis").copy(); cmap.set_bad("0.9")
    im = ax.imshow(np.ma.masked_invalid(heat.values.astype(float)), aspect="auto", cmap=cmap)
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            v = heat.values[i, j]
            ax.text(j, i, f"{v:.3f}" if np.isfinite(v) else "–", ha="center", va="center", fontsize=7,
                    color="white" if np.isfinite(v) and v < np.nanmax(heat.values) * 0.6 else "black")
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels([f"{short(b)} / {METHOD_LABELS[m]}" for b, m in heat.index])
    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_xticklabels(heat.columns, rotation=45, ha="right")
    ax.set_xlabel("Configuration")
    ax.set_ylabel("Processor / method")
    ax.set_title("$P_{valid}$ Across the Full Hardware Matrix (grey = pending / quarantined)")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("$P_{valid}$")
    plt.tight_layout()
    save_fig("FIG5_full_Pvalid_heatmap")
    plt.show()
else:
    print("Heatmap pending: no completed results.")

In [ ]:
# ============================================================
# 28. FIGURE 6 — HAGCS PARETO FRONT PER PROCESSOR AND n
# ============================================================
groups = [(b, n) for b in BACKENDS for n in N_VALUES]
elig = hagcs_df[hagcs_df["eligible"]]
if not elig.empty:
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    for ax, (bname, n) in zip(axes.ravel(), groups):
        g = elig[(elig.backend == bname) & (elig.n == n)]
        if g.empty:
            ax.text(0.5, 0.5, "pending", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(f"{short(bname)}, n={n}"); continue
        dom = g[~g.pareto_optimal]
        ax.errorbar(dom.two_qubit_gates, dom.P_valid, yerr=2 * dom.sigma_P, fmt="o", mfc="none",
                    color="0.55", capsize=2, label="Dominated")
        front = g[g.pareto_optimal].sort_values("two_qubit_gates")
        ax.errorbar(front.two_qubit_gates, front.P_valid, yerr=2 * front.sigma_P, fmt="-o",
                    color=BACKEND_COLORS[bname], capsize=2, label="Pareto-feasible")
        sel = selected_hagcs[(selected_hagcs.backend == bname) & (selected_hagcs.n == n)]
        mp = max_pvalid[(max_pvalid.backend == bname) & (max_pvalid.n == n)]
        if not sel.empty:
            ax.scatter(sel.two_qubit_gates, sel.P_valid, s=260, facecolors="none", edgecolors="k",
                       linewidths=1.5, label="HAGCS selected", zorder=4)
        if not mp.empty:
            ax.scatter(mp.two_qubit_gates, mp.P_valid, marker="*", s=160, color="goldenrod",
                       label="max $P_{valid}$", zorder=5)
        for _, r in g.iterrows():
            ax.annotate(f"O{r.oracle}/k{int(r.k)}/{METHOD_LABELS[r.method]}", (r.two_qubit_gates, r.P_valid),
                        xytext=(4, 4), textcoords="offset points", fontsize=7)
        ax.axhline(random_baseline(n), color="k", linestyle=":", linewidth=1, label="random M/N")
        ax.set_title(f"{short(bname)}, n={n}")
        ax.set_xlabel("Two-qubit gate count")
        ax.set_ylabel("$P_{valid}$ (±2σ)")
        ax.grid(alpha=0.25)
    axes.ravel()[0].legend(frameon=False, fontsize=7)
    fig.suptitle("HAGCS: Pareto Feasibility and Selected Configuration", y=1.01)
    plt.tight_layout()
    save_fig("FIG6_HAGCS_pareto_front")
    plt.show()
else:
    print("Pareto figure pending: resource-matched completed results required.")

## Final audit and result checklist

The notebook is paper-ready only when the audit confirms:
- all 8 algorithmic configurations are defined;
- both oracle **circuits** mark exactly the clause-enumerated solutions (checked on every basis state);
- the exact statevector of every Grover circuit matches the analytical formula;
- Stage 3 covers every processor × configuration and is scored from raw IBM counts;
- the nominal shot budget is 500 throughout the primary campaign;
- every strategy is readout-corrected (Stage-3 RAW excepted, and flagged);
- logical and physical two-qubit counts are explicitly distinguished;
- uncertainty is reported with a source appropriate to the measurement primitive;
- mitigation gain is computed against the same backend/configuration RAW result;
- HAGCS selection is made only from Pareto-feasible primary data;
- Stage 6 validates the selected configuration/method with fresh jobs that were not used for selection;
- no missing value is replaced by a fabricated estimate.

In [ ]:
# ============================================================
# 29. FINAL AUDIT REPORT
# ============================================================
stage3_complete = len(raw_reference_df) == len(BACKENDS) * len(CONFIGS)
stage3_source = ", ".join(sorted(raw_reference_df.scored_from.unique()))

truth_ok = bool(truth_table_validation["oracle_A_circuit_equals_2SAT"].eq(True).all()
                and truth_table_validation["oracle_B_circuit_equals_2SAT"].eq(True).all())
exact_ok = bool((ideal_df["P_exact_statevector"] - ideal_df["P_ideal"]).abs().max() < 1e-6)
shots_ok = SHOTS == 500 and INDEPENDENT_SHOTS == 500
completed = master_df[master_df.P_valid.notna()]
mitigation_done = int(completed.method.isin(set(METHODS) - {"RAW"}).sum())
selection_from_pareto = bool(selected_hagcs["pareto_optimal"].all()) if not selected_hagcs.empty else None
stage6_done = len(compare) if not compare.empty else 0

audit = [
    ("Expected backend/configuration cells", len(BACKENDS)*len(CONFIGS)),
    ("Expected backend/configuration/method cells", len(BACKENDS)*len(CONFIGS)*len(METHODS)),
    ("Active master rows", len(master_df)),
    ("Completed P_valid records", len(completed)),
    ("  of which Stage-3 RAW (not readout-corrected)", int((completed.data_source == "stage3_raw_characterization").sum())),
    ("  of which new QPU", int((completed.data_source == "new_qpu").sum())),
    ("Completed mitigation (non-RAW) records", mitigation_done),
    ("Pending P_valid records", int(master_df.P_valid.isna().sum())),
    ("Oracle circuit validation passed (A and B, all basis states)", truth_ok),
    ("Exact statevector = analytical formula", exact_ok),
    ("Nominal primary shots = 500", shots_ok),
    ("Readout correction enabled", READOUT_CORRECTION),
    ("Stage 3 complete (2 processors x 8 configurations)", stage3_complete),
    ("Stage 3 scored from", stage3_source),
    ("HAGCS selection made from Pareto-feasible set", selection_from_pareto),
    ("Stage-6 selected configurations", len(selected_hagcs)),
    ("Selections with 95% CI above random guessing", int(selected_hagcs["above_random_baseline"].fillna(False).astype(bool).sum())),
    ("Stage-6 completed validation rows", stage6_done),
    ("Stage 4 mitigation claims supported by data", mitigation_done > 0),
    ("Stage 6 claims supported by data", stage6_done > 0),
]
audit_df = pd.DataFrame(audit, columns=["check", "value"])
display(audit_df)
audit_df.to_csv(TAB_DIR / "FINAL_audit_report.csv", index=False)

manifest = {
    "backends": BACKENDS, "n_values": N_VALUES, "oracles": ORACLES, "k_values": K_VALUES,
    "methods": METHODS, "hagcs_methods": HAGCS_METHODS, "nominal_shots": SHOTS,
    "independent_shots": INDEPENDENT_SHOTS, "readout_correction": READOUT_CORRECTION,
    "cal_shots": CAL_SHOTS, "zne_noise_factors": ZNE_NOISE_FACTORS,
    "trex_randomizations": TREX_RANDOMIZATIONS, "stage3_job_ids": STAGE3_JOB_IDS,
    "bit_order_convention": "x_i on data qubit i-1; solution strings are Qiskit count keys q_{n-1}..q_0",
    "solutions_qiskit_order": {str(n): KSAT[n]["solutions"] for n in N_VALUES},
    "solutions_variable_order": {str(n): [assignment_string(s) for s in KSAT[n]["solutions"]] for n in N_VALUES},
    "hagcs_pareto_objectives": HAGCS_PARETO_OBJECTIVES, "require_above_random": REQUIRE_ABOVE_RANDOM,
    "artifacts": {
        "truth_table": str(TAB_DIR / "stage1_oracle_truth_table_validation.csv"),
        "master_matrix": str(TAB_DIR / "stage4_master_result_matrix.csv"),
        "hagcs_metrics": str(TAB_DIR / "stage5_HAGCS_complete_metrics.csv"),
        "selected_hagcs": str(TAB_DIR / "stage5_HAGCS_selected_for_validation.csv"),
        "hagcs_tradeoff": str(TAB_DIR / "stage5_HAGCS_vs_maxP_tradeoff.csv"),
        "stage6_validation": str(TAB_DIR / "stage6_independent_validation_results.csv"),
        "audit": str(TAB_DIR / "FINAL_audit_report.csv"),
    },
}
(ROOT / "FINAL_experiment_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Final audit manifest written.")

## Consolidated six-stage evaluation matrix

The following cell exports one machine-readable summary describing what is available at each stage and which metrics are primary, with the live completion status of each stage.

In [ ]:
# ============================================================
# 30. SIX-STAGE EVALUATION MATRIX / ARTIFACT INDEX
# ============================================================
n_stage3 = int((master_df.data_source == "stage3_raw_characterization").sum())
stage_rows = [
 {"stage":1,"name":"Circuit and configuration space","scope":"n={5,6} × Oracle={A,B} × k={5,6}",
  "primary_metrics":"circuit-level oracle equivalence, N, M, logical 2Q/depth",
  "status":"complete" if truth_ok else "FAILED","artifact":str(TAB_DIR/"stage1_circuit_resources.csv")},
 {"stage":2,"name":"Ideal simulation","scope":"8 configurations",
  "primary_metrics":"P_ideal (analytic = exact statevector), Aer P_valid, expected shots",
  "status":"complete" if exact_ok else "FAILED","artifact":str(TAB_DIR/"stage2_ideal_simulation.csv")},
 {"stage":3,"name":"2-SAT hardware characterization","scope":"2 processors × 8 configurations",
  "primary_metrics":"P_valid, physical 2Q gates, depth, uncertainty",
  "status":f"{len(raw_reference_df)}/16 cells from IBM jobs ({stage3_source})","artifact":str(TAB_DIR/"stage3_raw_hardware_characterization.csv")},
 {"stage":4,"name":"Error suppression and mitigation","scope":"2 processors × 8 configurations × 6 methods",
  "primary_metrics":"P_valid, uncertainty, mitigation gain, overhead, QPU cost",
  "status":f"{mitigation_done}/80 non-RAW cells","artifact":str(TAB_DIR/"stage4_master_result_matrix.csv")},
 {"stage":5,"name":"HAGCS selection","scope":"backend × n",
  "primary_metrics":"Pareto feasibility, then J_robust; trade-off vs max-P_valid",
  "status":f"{len(selected_hagcs)} selections","artifact":str(TAB_DIR/"stage5_HAGCS_selected_for_validation.csv")},
 {"stage":6,"name":"Independent validation","scope":"selected configuration/method per backend × n",
  "primary_metrics":"fresh P_valid, CI95, deviation from selection",
  "status":f"{stage6_done}/{len(selected_hagcs)} validated","artifact":str(TAB_DIR/"stage6_independent_validation_results.csv")},
]
six_stage_df = pd.DataFrame(stage_rows)
six_stage_df.to_csv(TAB_DIR / "SIX_STAGE_EVALUATION_MATRIX.csv", index=False)
display(six_stage_df)

## Notes for the manuscript

1. The ideal simulator is the **noiseless algorithmic reference**, not a hardware baseline.
2. Oracle A is a direct marked-state realization; Oracle B is a clause-structured 2-SAT realization. The notebook verifies, at circuit level and on every basis state, that both mark exactly the satisfying assignments.
3. **Bit order.** Variable x_i is data qubit i−1 and solution strings are Qiskit count keys. The n=6 satisfying set is `{000000, 101010, 111111}` as count keys, i.e. `{000000, 010101, 111111}` as assignments x1…x6. State which order a table uses.
4. Stage 3 (jobs `db1l4n9b694s73dsemrg`, `db1l4p6egvvc73bhpq40`) is characterization data used for selection. It must not be reported as the independent validation; only fresh Stage-6 jobs are.
5. DD and PT are reported as suppression/twirling methods; ZNE and TREX are expectation-value mitigation methods. All strategies are readout-corrected; the Stage-3 RAW characterization is not, and mitigation gains against it are flagged `gain_like_for_like = False`.
6. ZNE/TREX/ALL4 `P_valid` values are estimator expectation values; their uncertainty is not replaced by a binomial approximation.
7. HAGCS is presented as the study's proposed selection framework: Pareto feasibility filtering, then uncertainty-aware, cost-sensitive ranking. Its score should be defined explicitly in the manuscript.
8. The HAGCS-vs-max-P_valid trade-off (e.g. fewer two-qubit gates at a ΔP_valid within uncertainty) is a property of the selection on primary data. Stage 6 is the fresh validation and is reported only from completed Stage-6 jobs.
9. For n=5, k=5 the ideal P_valid (0.077) is below random guessing (0.094). Most raw hardware values lie near M/N; report the random baseline next to every P_valid and do not interpret values indistinguishable from it as Grover amplification.
10. Do not claim quantum advantage. The supported claim is hardware-aware characterization and resource/quality-aware configuration selection.